In [ ]:
## ----------- SOTA MODELS -----------
## This module compares other state-of-the-art models of multilingual multitask model
## We used BERT, mBERT, DistilBERT, mDistilBERT, and XML-Roberta
## We choose those models because of their performance on the multilingual dataset.
## Change the model_name and its tokenizer when you need to run the specific model
## 
## The SOTA models employ feedback (with emoji), no imbalanced dataset strategy, and no Emoji Attention
## We define the original models without any treatment or improvement made to our approach.



import os
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.optim import AdamW
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    accuracy_score, roc_auc_score, precision_recall_fscore_support
)
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

# =================== BASELINES ===================
# # BERT / mBERT (bert-base multilingual)
# from transformers import BertTokenizer, BertModel
# # tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# tokenizer = BertTokenizer.from_pretrained('bert-base-multilingual-cased')

# # DistilBertModel
# from transformers import DistilBertTokenizer, DistilBertModel
# tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

# # DistilBertModel-Multilingual
# from transformers import DistilBertTokenizer, DistilBertModel
# tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-multilingual-cased")

# # LaBSE
# from transformers import AutoTokenizer, AutoModel
# tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/LaBSE")

# XML-Roberta
from transformers import XLMRobertaTokenizer, XLMRobertaModel
tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

# ================== KONFIGURASI ==================
CHECKPOINT_DIR = "outputModel/BASELINE/XML-Roberta"
BEST_NAME_MODEL = "best_model_multi_task.pt"
BEST_CKPT_PATH = os.path.join(CHECKPOINT_DIR, BEST_NAME_MODEL)
TRAIN_CSV_PATH = "dataset/4. newlabel/extend/train.csv"
TEST_CSV_PATH = "dataset/4. newlabel/extend/test.csv"
OUTPUT_PRED_BEST = "outputPrediksi/XML-Roberta/pred_feedbackEmo.csv"

KFOLD_SPLITS = 10
MAX_LEN = 256
BATCH_SIZE = 8
THRESHOLD = 0.5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EPOCH = 10
DROPOUT = 0.1
LR = 5.326347317195311e-05

EMOTION_LABELS = ['anger','anticipation','disgust','fear','joy','sadness', 'surprise', 'trust']
NUM_EMOTIONS = len(EMOTION_LABELS)
NUM_SENTIMENT = 3
NUM_ASPECT = 5

# ================== MODEL ==================
class MultiTask(nn.Module):
    def __init__(self, num_emotions=NUM_EMOTIONS, num_sentiment=NUM_SENTIMENT, num_aspect=NUM_ASPECT, dropout=DROPOUT):
        super().__init__()
        # self.backbone = BertModel.from_pretrained('bert-base-multilingual-cased')
        self.backbone = XLMRobertaModel.from_pretrained("xlm-roberta-base", return_dict=True)
        # self.backbone = AutoModel.from_pretrained("sentence-transformers/LaBSE")
        # self.backbone = DistilBertModel.from_pretrained("distilbert-base-multilingual-cased")
        self.dropout = nn.Dropout(dropout)
        self.classifier_emotion = nn.Linear(self.backbone.config.hidden_size, num_emotions)
        self.classifier_sentiment = nn.Linear(self.backbone.config.hidden_size, num_sentiment)
        self.classifier_aspect = nn.Linear(self.backbone.config.hidden_size, num_aspect)

    def forward(self, input_ids, attention_mask):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        x = self.dropout(pooled)
        logits_emotion = self.classifier_emotion(x)
        logits_sentiment = self.classifier_sentiment(x)
        logits_aspect = self.classifier_aspect(x)
        return logits_emotion, logits_sentiment, logits_aspect

def batch_encode(texts, max_len=MAX_LEN):
    return tokenizer(
        texts,
        add_special_tokens=True,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_attention_mask=True,
        return_tensors='pt'
    )

# ================== DATASET ==================
class MultiTaskDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts = df['feedback'].tolist()
        self.emotions = df[EMOTION_LABELS].values.astype(float)
        self.sentiments = df['sentiment'].astype(int).values
        self.aspects = df['aspect'].astype(int).values
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'emotion_labels': torch.tensor(self.emotions[idx], dtype=torch.float),
            'sentiment_label': torch.tensor(self.sentiments[idx], dtype=torch.long),
            'aspect_label': torch.tensor(self.aspects[idx], dtype=torch.long)
        }

def create_data_loaders(csv_path, tokenizer, max_len, batch_size, seed=42):
    df = pd.read_csv(csv_path)
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=seed)
    train_ds = MultiTaskDataset(train_df, tokenizer, max_len)
    val_ds   = MultiTaskDataset(val_df, tokenizer, max_len)
    return (
        DataLoader(train_ds, shuffle=True,  batch_size=batch_size),
        DataLoader(val_ds,   shuffle=False, batch_size=batch_size)
    )

# ================== TRAINING ==================
def train_model(model, train_loader, val_loader, device, epochs=EPOCH, lr=LR, checkpoint_dir=CHECKPOINT_DIR):
    os.makedirs(checkpoint_dir, exist_ok=True)
    optimizer = AdamW(model.parameters(), lr=lr)
    loss_fn_emotion = nn.BCEWithLogitsLoss()
    loss_fn_sentiment = nn.CrossEntropyLoss()
    loss_fn_aspect = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch} — Training"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            emotion_labels = batch['emotion_labels'].to(device)
            sentiment_label = batch['sentiment_label'].to(device)
            aspect_label = batch['aspect_label'].to(device) 

            logits_emotion, logits_sentiment, logits_aspect = model(input_ids, attention_mask)
            loss_emotion = loss_fn_emotion(logits_emotion, emotion_labels)
            loss_sentiment = loss_fn_sentiment(logits_sentiment, sentiment_label)
            loss_aspect = loss_fn_aspect(logits_aspect, aspect_label)
            loss = loss_emotion + loss_sentiment + loss_aspect
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_train_loss = total_loss / len(train_loader)

        model.eval()
        total_val_loss = 0
        total_val_emotion = 0
        total_val_sentiment = 0
        total_val_aspect = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch} — Validating"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                emotion_labels = batch['emotion_labels'].to(device)
                sentiment_label = batch['sentiment_label'].to(device)
                aspect_label = batch['aspect_label'].to(device)

                logits_emotion, logits_sentiment, logits_aspect = model(input_ids, attention_mask)
                loss_emotion = loss_fn_emotion(logits_emotion, emotion_labels)
                loss_sentiment = loss_fn_sentiment(logits_sentiment, sentiment_label)
                loss_aspect = loss_fn_aspect(logits_aspect, aspect_label)
                loss = loss_emotion + loss_sentiment + loss_aspect
                
                total_val_loss += loss.item()
                total_val_emotion += loss_emotion.item()
                total_val_sentiment += loss_sentiment.item()
                total_val_aspect += loss_aspect.item()
                
        avg_val_loss = total_val_loss / len(val_loader)
        avg_val_emotion = total_val_emotion / len(val_loader)
        avg_val_sentiment = total_val_sentiment / len(val_loader)
        avg_val_aspect = total_val_aspect / len(val_loader)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save({
                'epoch': epoch,
                'state_dict': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'valid_loss_min': avg_val_loss
            }, os.path.join(checkpoint_dir, BEST_NAME_MODEL))

        print(f"Epoch {epoch} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} "
              f"| Emotion Val Loss: {avg_val_emotion:.4f} | Sentiment Val Loss: {avg_val_sentiment:.4f} "
              f"| Aspect Val Loss: {avg_val_aspect:.4f} | Best Val Loss: {best_val_loss:.4f}")

# ================== METRICS DETAIL EMOTION ==================
def evaluate_multilabel(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    subset_acc = accuracy_score(y_true, y_pred)
    sample_acc = np.mean((y_true == y_pred).sum(axis=1) / y_true.shape[1])
    micro_prec = precision_score(y_true, y_pred, average='micro', zero_division=0)
    micro_rec  = recall_score(y_true, y_pred, average='micro', zero_division=0)
    micro_f1   = f1_score(y_true, y_pred, average='micro', zero_division=0)
    macro_prec = precision_score(y_true, y_pred, average='macro', zero_division=0)
    macro_rec  = recall_score(y_true, y_pred, average='macro', zero_division=0)
    macro_f1   = f1_score(y_true, y_pred, average='macro', zero_division=0)
    per_label_prec = precision_score(y_true, y_pred, average=None, zero_division=0)
    per_label_rec  = recall_score(y_true, y_pred, average=None, zero_division=0)
    per_label_f1   = f1_score(y_true, y_pred, average=None, zero_division=0)
    per_label_auc = []
    for j in range(y_true.shape[1]):
        y_col = y_true[:, j]
        if len(np.unique(y_col)) == 2:
            try:
                auc = roc_auc_score(y_col, y_prob[:, j])
            except ValueError:
                auc = np.nan
        else:
            auc = np.nan
        per_label_auc.append(auc)
    valid_aucs = [a for a in per_label_auc if not np.isnan(a)]
    macro_auc = np.mean(valid_aucs) if len(valid_aucs) > 0 else np.nan
    metrics = {
        "subset_accuracy": subset_acc,
        "sample_accuracy": sample_acc,
        "micro_precision": micro_prec,
        "micro_recall": micro_rec,
        "micro_f1": micro_f1,
        "macro_precision": macro_prec,
        "macro_recall": macro_rec,
        "macro_f1": macro_f1,
        "macro_auc": macro_auc,
        "per_label_precision": per_label_prec,
        "per_label_recall": per_label_rec,
        "per_label_f1": per_label_f1,
        "per_label_auc": per_label_auc
    }
    return metrics

def print_metrics(metrics, label_names):
    print("\n===== EMOTION EVALUATION METRICS =====")
    print(f"Subset Accuracy   : {metrics['subset_accuracy']:.4f}")
    print(f"Sample Accuracy   : {metrics['sample_accuracy']:.4f}")
    print(f"Micro Precision   : {metrics['micro_precision']:.4f}")
    print(f"Micro Recall      : {metrics['micro_recall']:.4f}")
    print(f"Micro F1          : {metrics['micro_f1']:.4f}")
    print(f"Macro Precision   : {metrics['macro_precision']:.4f}")
    print(f"Macro Recall      : {metrics['macro_recall']:.4f}")
    print(f"Macro F1          : {metrics['macro_f1']:.4f}")
    print(f"Macro ROC-AUC     : {metrics['macro_auc']:.4f}" if metrics['macro_auc']==metrics['macro_auc'] else "Macro ROC-AUC     : NaN")
    print("\nPer-Label Metrics:")
    for i, name in enumerate(label_names):
        auc_val = metrics['per_label_auc'][i]
        auc_str = f"{auc_val:.4f}" if auc_val == auc_val else "NaN"
        print(f"- {name:22s} | P: {metrics['per_label_precision'][i]:.4f} "
              f"| R: {metrics['per_label_recall'][i]:.4f} "
              f"| F1: {metrics['per_label_f1'][i]:.4f} "
              f"| AUC: {auc_str}")

# ================== EVALUASI ==================
def evaluate_multitask(y_true_emotion, y_prob_emotion, y_true_sentiment, y_pred_sentiment, y_true_aspect, y_pred_aspect):
    # EMOTION
    emotion_metrics = evaluate_multilabel(y_true_emotion, y_prob_emotion, threshold=THRESHOLD)
    print_metrics(emotion_metrics, EMOTION_LABELS)
    
    # SENTIMENT
    acc = accuracy_score(y_true_sentiment, y_pred_sentiment)
    prec, rec, f1, _ = precision_recall_fscore_support(y_true_sentiment, y_pred_sentiment, average='macro')

    print("\n===== SENTIMENT (MULTI-CLASS) =====")
    print(f"Accuracy          : {acc:.4f}")
    print(f"Precision         : {prec:.4f}, Recall: {rec:.4f}, F1: {f1:.4f}")

    # ASPECT
    acc_aspect = accuracy_score(y_true_aspect, y_pred_aspect)
    prec_aspect, rec_aspect, f1_aspect, _ = precision_recall_fscore_support(y_true_aspect, y_pred_aspect, average='macro')

    print("\n===== ASPECT (MULTI-CLASS) =====")
    print(f"Accuracy          : {acc_aspect:.4f}")
    print(f"Precision         : {prec_aspect:.4f}, Recall: {rec_aspect:.4f}, F1: {f1_aspect:.4f}")

# ================== MAIN ==================
def main():
    train_loader, val_loader = create_data_loaders(TRAIN_CSV_PATH, tokenizer, MAX_LEN, BATCH_SIZE)
    model = MultiTask().to(DEVICE)
    train_model(model, train_loader, val_loader, DEVICE)

    print("\n[INFERENSI DAN EVALUASI]")
    test_df = pd.read_csv(TEST_CSV_PATH)
    y_true_emotion = test_df[EMOTION_LABELS].values.astype(int)
    y_true_sentiment = test_df['sentiment'].astype(int).values
    y_true_aspect = test_df['aspect'].astype(int).values
    model.load_state_dict(torch.load(BEST_CKPT_PATH, map_location=DEVICE)['state_dict'])
    model.eval()

    probs_emotion = []
    preds_sentiment = []
    preds_aspect = []
    with torch.no_grad():
        for i in tqdm(range(0, len(test_df), BATCH_SIZE), desc="Predicting"):
            texts = test_df['feedback'].iloc[i:i+BATCH_SIZE].tolist()
            enc = batch_encode(texts)
            input_ids = enc['input_ids'].to(DEVICE)
            attention_mask = enc['attention_mask'].to(DEVICE)
            logits_emotion, logits_sentiment, logits_aspect = model(input_ids, attention_mask)
            probs_emotion.append(torch.sigmoid(logits_emotion).cpu().numpy())
            preds_sentiment.extend(torch.argmax(logits_sentiment, dim=1).cpu().numpy())
            preds_aspect.extend(torch.argmax(logits_aspect, dim=1).cpu().numpy())

    probs_emotion = np.vstack(probs_emotion)
    evaluate_multitask(y_true_emotion, probs_emotion, y_true_sentiment, preds_sentiment, y_true_aspect, preds_aspect)

    out_df = test_df.copy()
    for i, lab in enumerate(EMOTION_LABELS):
        out_df[f"pred_{lab}"] = (probs_emotion[:, i] >= THRESHOLD).astype(int)
    out_df['pred_sentiment'] = preds_sentiment
    out_df['pred_aspect'] = preds_aspect
    os.makedirs(os.path.dirname(OUTPUT_PRED_BEST), exist_ok=True)
    out_df.to_csv(OUTPUT_PRED_BEST, index=False)
    print(f"Prediksi disimpan ke {OUTPUT_PRED_BEST}")

from sklearn.model_selection import KFold

def train_one_fold(fold, model, dataset, kf_indices):
    train_idx, val_idx = kf_indices[fold]
    train_ds = Subset(dataset, train_idx)
    val_ds   = Subset(dataset, val_idx)
    train_loader = DataLoader(train_ds, shuffle=True, batch_size=BATCH_SIZE)
    val_loader   = DataLoader(val_ds, shuffle=False, batch_size=BATCH_SIZE)

    optimizer = AdamW(model.parameters(), lr=LR)
    loss_fn_emotion = nn.BCEWithLogitsLoss()
    loss_fn_sentiment = nn.CrossEntropyLoss()
    loss_fn_aspect = nn.CrossEntropyLoss()

    best_val_loss = float('inf')
    for epoch in range(1, EPOCH + 1):
        model.train()
        for batch in tqdm(train_loader, desc=f"Epoch {epoch} - Training"):
            input_ids = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            emotion_labels = batch['emotion_labels'].to(DEVICE)
            sentiment_label = batch['sentiment_label'].to(DEVICE)
            aspect_label = batch['aspect_label'].to(DEVICE)

            logits_emotion, logits_sentiment, logits_aspect = model(input_ids, attention_mask)
            loss = (
                loss_fn_emotion(logits_emotion, emotion_labels) +
                loss_fn_sentiment(logits_sentiment, sentiment_label) +
                loss_fn_aspect(logits_aspect, aspect_label)
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        model.eval()
        total_val_loss = 0
        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"Epoch {epoch} - Validating"):
                input_ids = batch['input_ids'].to(DEVICE)
                attention_mask = batch['attention_mask'].to(DEVICE)
                emotion_labels = batch['emotion_labels'].to(DEVICE)
                sentiment_label = batch['sentiment_label'].to(DEVICE)
                aspect_label = batch['aspect_label'].to(DEVICE)

                logits_emotion, logits_sentiment, logits_aspect = model(input_ids, attention_mask)
                loss = (
                    loss_fn_emotion(logits_emotion, emotion_labels) +
                    loss_fn_sentiment(logits_sentiment, sentiment_label) +
                    loss_fn_aspect(logits_aspect, aspect_label)
                )
                total_val_loss += loss.item()

        avg_val_loss = total_val_loss / len(val_loader)
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), os.path.join(CHECKPOINT_DIR, f"{BEST_NAME_MODEL}_fold{fold+1}.pt"))


def run_kfold_training():
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    df = pd.read_csv(TRAIN_CSV_PATH)
    dataset = MultiTaskDataset(df, tokenizer, MAX_LEN)
    kf = KFold(n_splits=KFOLD_SPLITS, shuffle=True, random_state=42)
    kf_indices = list(kf.split(np.arange(len(dataset))))

    for fold in range(KFOLD_SPLITS):
        print(f"\n================ Fold {fold+1}/{KFOLD_SPLITS} ================")
        model = MultiTask().to(DEVICE)
        train_one_fold(fold, model, dataset, kf_indices)


def predict_ensemble():
    test_df = pd.read_csv(TEST_CSV_PATH)
    y_true_emotion = test_df[EMOTION_LABELS].values.astype(int)
    y_true_sentiment = test_df['sentiment'].astype(int).values
    y_true_aspect = test_df['aspect'].astype(int).values

    probs_emotion = []
    logits_sentiment = []
    logits_aspect = []

    with torch.no_grad():
        for fold in range(KFOLD_SPLITS):
            print(f"Inferencing with Fold {fold+1} model")
            model = MultiTask().to(DEVICE)
            model.load_state_dict(torch.load(os.path.join(CHECKPOINT_DIR, f"{BEST_NAME_MODEL}_fold{fold+1}.pt"), map_location=DEVICE))
            model.eval()

            fold_probs_emotion = []
            fold_logits_sentiment = []
            fold_logits_aspect = []

            for i in tqdm(range(0, len(test_df), BATCH_SIZE)):
                texts = test_df['feedback'].iloc[i:i+BATCH_SIZE].tolist()
                enc = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=MAX_LEN)
                input_ids = enc['input_ids'].to(DEVICE)
                attention_mask = enc['attention_mask'].to(DEVICE)

                logit_emotion, logit_sentiment, logit_aspect = model(input_ids, attention_mask)
                fold_probs_emotion.append(torch.sigmoid(logit_emotion).cpu().numpy())
                fold_logits_sentiment.append(logit_sentiment.cpu().numpy())
                fold_logits_aspect.append(logit_aspect.cpu().numpy())

            probs_emotion.append(np.vstack(fold_probs_emotion))
            logits_sentiment.append(np.vstack(fold_logits_sentiment))
            logits_aspect.append(np.vstack(fold_logits_aspect))

    avg_probs_emotion = np.mean(probs_emotion, axis=0)
    avg_sentiment = np.mean(logits_sentiment, axis=0)
    avg_aspect = np.mean(logits_aspect, axis=0)

    pred_sentiment = np.argmax(avg_sentiment, axis=1)
    pred_aspect = np.argmax(avg_aspect, axis=1)

    evaluate_multitask(
        y_true_emotion, avg_probs_emotion,
        y_true_sentiment, pred_sentiment,
        y_true_aspect, pred_aspect
    )

    out_df = test_df.copy()
    for i, lab in enumerate(EMOTION_LABELS):
        out_df[f"pred_{lab}"] = (avg_probs_emotion[:, i] >= THRESHOLD).astype(int)
    out_df['pred_sentiment'] = pred_sentiment
    out_df['pred_aspect'] = pred_aspect
    os.makedirs(os.path.dirname(OUTPUT_PRED_BEST), exist_ok=True)
    out_df.to_csv(OUTPUT_PRED_BEST, index=False)
    print(f"Prediksi disimpan ke {OUTPUT_PRED_BEST}")

if __name__ == "__main__":
    # main()
    run_kfold_training()
    print("\n[INFERENSI DAN EVALUASI ENSEMBLE K-FOLD]")
    predict_ensemble()
